## Faiss test
- OpenAI API를 이용해서 모델 작동 테스트

In [29]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from dotenv import load_dotenv
from langchain.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain.chains import LLMChain
from langchain_core.output_parsers import StrOutputParser


# 환경 변수 로드
load_dotenv()

folder_path = "./data/"
pkl_path = "./data/index.pkl"

embeddings = OpenAIEmbeddings(model = "text-embedding-3-small")  # OpenAI API 키는 .env 파일에서 로드됨

db = FAISS.load_local(
        folder_path=folder_path,
        embeddings=embeddings,
        allow_dangerous_deserialization=True
    )
retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": 3})

llm = ChatOpenAI(
        model="gpt-4o",  # OpenAI 모델 선택
        temperature=0,  # 출력의 일관성을 높이기 위해 온도 설정
        max_tokens=1024)

prompt_template = """
You are an expert in infectious diseases. Provide accurate answers based strictly on the given context.

Instructions:
- Answer only using the provided context. Do not include creative ideas or answers not directly related to the question.
- If the question is outside the context, respond with: "I don't know."
- Include references in the "References" section using the source's URL from the metadata.
- Answer in Korean

#Example Format:
    (detailed answer to the question)

    **출처**
    - (URL of the source)

#Context:
{context}

#Question:
{question}

#Answer:
"""
prompt = PromptTemplate(input_variables=["context", "question"], template=prompt_template)

# 5. LLM 체인 생성
llm_chain = prompt|llm

# 6. RAG 체인 구성
rag_chain = {"context": retriever, "question": RunnablePassthrough()} | llm_chain |StrOutputParser()

In [30]:
db.similarity_search('바이러스출혈열은?')

[Document(id='f5d2f632-d4ae-4996-9bba-366e8522e87f', metadata={'source': 'https://041202.busancidc-html.com', 'title': '바이러스성 출혈열 질병 개요 ', 'language': 'ko'}, page_content='바이러스성 출혈열 질병 개요 - 성기단순포진 감염 발생에 관한 정보 -구 분내 용발생현황[국외현황]◦ 전\xa0세계 모든 인구집단에서 발견됨-\xa0(Ⅰ형)\xa0어린이나 젊은 성인에서 주로 잇몸구내염이나 인두염으로 초회 감염을 일으킴-\xa0(Ⅱ형)\xa0성기단순포진으로 초회 감염을 일으키며 사춘기 이후 성 활동이 시작되면서\xa0발생이 증가함・15-49세 연령에서 HSV 2형의 유병률은 11.3%로 추정・아프리카 지역에서의 발생이 가장 많고 아메리카, 동부 지중해, 유럽 지역에서의\xa0발생은 상대적으로 적음・전 지역에서 남성보다 여성에서 발생이 더 많음\xa0* (남성→여성) 11-17%, (여성→남성) 3-4%[국내현황]◦ 2001년 표본감시감염병으로 지정됨◦ 2010년 1,572건에서 2019년 11,608건으로 증가 추세◦ 2022년 10,403건으로 감소함<참고>◦\xa0성기단순포진은 한번 감염되면 평생 잠복감염 상태로 재발이 반복되는 질환임◦\xa0완치가 불가능하여 항바이러스제 치료 후에도 평생 바이러스가 몸에 존재하기 때문에\xa0다른 성매개감염과 달리 연령이 증가할수록 유병률이 높아지는 특성이 있음◦\xa0성기단순포진의 역학적 자료 해석 시 고령환자의 분포가 많은 것을 높은 발생률로 해석하는 오류를 범하지 말아야 함'),
 Document(id='8d00a00d-91f6-4b08-8180-b508207c7e8f', metadata={'source': 'https://045501.busancidc-html.com', 'title': '질병개요 ', 'language': 'No language found.'}, page_content='질병개요

In [32]:
question = "남아메리카출혈열 발생원인은?"

response = rag_chain.invoke(question)
print(response)

I don't know.


In [35]:
from langchain_community.vectorstores import FAISS
from langchain.embeddings import OpenAIEmbeddings
# FAISS DB 파일 경로
faiss_folder_path = "./data/"
embedding_model = OpenAIEmbeddings(model="text-embedding-ada-002")  # 사용한 임베딩 모델

# FAISS 인덱스 로드
db = FAISS.load_local(folder_path=faiss_folder_path, embeddings=embedding_model,allow_dangerous_deserialization=True)

# FAISS 인덱스의 차원 확인
print(f"FAISS DB에 저장된 임베딩의 차원: {db.index.d}")

FAISS DB에 저장된 임베딩의 차원: 1536


### FAISS DB 의 임베딩 차원이 너무 낮은거 같음... 그래서 OpenAIEmbedding의 text-embedding-3-large 모델을 사용해서 다시 DB를 생성

In [37]:
from langchain_community.document_loaders import WebBaseLoader
from dotenv import load_dotenv
from langchain_teddynote import logging
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

load_dotenv()

True

In [42]:
urls = [
"https://010601.busancidc-html.com",
"https://010602.busancidc-html.com",
"https://010603.busancidc-html.com",
"https://010604.busancidc-html.com",
"https://010605.busancidc-html.com",
"https://010606.busancidc-html.com",
"https://010607.busancidc-html.com",
"https://010608.busancidc-html.com",
"https://010609.busancidc-html.com",
"https://010610.busancidc-html.com",
"https://012001.busancidc-html.com",
"https://012002.busancidc-html.com",
"https://012003.busancidc-html.com",
"https://012004.busancidc-html.com",
"https://012005.busancidc-html.com",
"https://012006.busancidc-html.com",
"https://012007.busancidc-html.com",
"https://012008.busancidc-html.com",
"https://012010.busancidc-html.com",
"https://012201.busancidc-html.com",
"https://012202.busancidc-html.com",
"https://012203.busancidc-html.com",
"https://012205.busancidc-html.com",
"https://012206.busancidc-html.com",
"https://012207.busancidc-html.com",
"https://012210.busancidc-html.com",
"https://011201.busancidc-html.com",
"https://011202.busancidc-html.com",
"https://011203.busancidc-html.com",
"https://011204.busancidc-html.com",
"https://011205.busancidc-html.com",
"https://011206.busancidc-html.com",
"https://011207.busancidc-html.com",
"https://011208.busancidc-html.com",
"https://011209.busancidc-html.com",
"https://011210.busancidc-html.com",
"https://012401.busancidc-html.com",
"https://012402.busancidc-html.com",
"https://012403.busancidc-html.com",
"https://012404.busancidc-html.com",
"https://012405.busancidc-html.com",
"https://012406.busancidc-html.com",
"https://012407.busancidc-html.com",
"https://012408.busancidc-html.com",
"https://012409.busancidc-html.com",
"https://012410.busancidc-html.com",
"https://010401.busancidc-html.com",
"https://010402.busancidc-html.com",
"https://010403.busancidc-html.com",
"https://010404.busancidc-html.com",
"https://010405.busancidc-html.com",
"https://010406.busancidc-html.com",
"https://010407.busancidc-html.com",
"https://010408.busancidc-html.com",
"https://010409.busancidc-html.com",
"https://010410.busancidc-html.com",
"https://011101.busancidc-html.com",
"https://011102.busancidc-html.com",
"https://011103.busancidc-html.com",
"https://011104.busancidc-html.com",
"https://011105.busancidc-html.com",
"https://011106.busancidc-html.com",
"https://011107.busancidc-html.com",
"https://011108.busancidc-html.com",
"https://011109.busancidc-html.com",
"https://011110.busancidc-html.com",
"https://010301.busancidc-html.com",
"https://010302.busancidc-html.com",
"https://010303.busancidc-html.com",
"https://010304.busancidc-html.com",
"https://010305.busancidc-html.com",
"https://010306.busancidc-html.com",
"https://010307.busancidc-html.com",
"https://010308.busancidc-html.com",
"https://010309.busancidc-html.com",
"https://010310.busancidc-html.com",
"https://010101.busancidc-html.com",
"https://010102.busancidc-html.com",
"https://010103.busancidc-html.com",
"https://010104.busancidc-html.com",
"https://010105.busancidc-html.com",
"https://010106.busancidc-html.com",
"https://010107.busancidc-html.com",
"https://010108.busancidc-html.com",
"https://010109.busancidc-html.com",
"https://010110.busancidc-html.com",
"https://010901.busancidc-html.com",
"https://010902.busancidc-html.com",
"https://010903.busancidc-html.com",
"https://010904.busancidc-html.com",
"https://010905.busancidc-html.com",
"https://010906.busancidc-html.com",
"https://010907.busancidc-html.com",
"https://010908.busancidc-html.com",
"https://010909.busancidc-html.com",
"https://010910.busancidc-html.com",
"https://011501.busancidc-html.com",
"https://011502.busancidc-html.com",
"https://011503.busancidc-html.com",
"https://011504.busancidc-html.com",
"https://011505.busancidc-html.com",
"https://011506.busancidc-html.com",
"https://011507.busancidc-html.com",
"https://011508.busancidc-html.com",
"https://011509.busancidc-html.com",
"https://011510.busancidc-html.com",
"https://010801.busancidc-html.com",
"https://010802.busancidc-html.com",
"https://010803.busancidc-html.com",
"https://010804.busancidc-html.com",
"https://010805.busancidc-html.com",
"https://010806.busancidc-html.com",
"https://010807.busancidc-html.com",
"https://010808.busancidc-html.com",
"https://010809.busancidc-html.com",
"https://010810.busancidc-html.com",
"https://011001.busancidc-html.com",
"https://011002.busancidc-html.com",
"https://011003.busancidc-html.com",
"https://011004.busancidc-html.com",
"https://011005.busancidc-html.com",
"https://011006.busancidc-html.com",
"https://011007.busancidc-html.com",
"https://011008.busancidc-html.com",
"https://011009.busancidc-html.com",
"https://011010.busancidc-html.com",
"https://011701.busancidc-html.com",
"https://011703.busancidc-html.com",
"https://011704.busancidc-html.com",
"https://011706.busancidc-html.com",
"https://011710.busancidc-html.com",
"https://012301.busancidc-html.com",
"https://012302.busancidc-html.com",
"https://012303.busancidc-html.com",
"https://012304.busancidc-html.com",
"https://012305.busancidc-html.com",
"https://012306.busancidc-html.com",
"https://012307.busancidc-html.com",
"https://012308.busancidc-html.com",
"https://012309.busancidc-html.com",
"https://012310.busancidc-html.com",
"https://010701.busancidc-html.com",
"https://010702.busancidc-html.com",
"https://010703.busancidc-html.com",
"https://010704.busancidc-html.com",
"https://010705.busancidc-html.com",
"https://010706.busancidc-html.com",
"https://010707.busancidc-html.com",
"https://010708.busancidc-html.com",
"https://010709.busancidc-html.com",
"https://010710.busancidc-html.com",
"https://011601.busancidc-html.com",
"https://011602.busancidc-html.com",
"https://011603.busancidc-html.com",
"https://011604.busancidc-html.com",
"https://011605.busancidc-html.com",
"https://011606.busancidc-html.com",
"https://011607.busancidc-html.com",
"https://011608.busancidc-html.com",
"https://011609.busancidc-html.com",
"https://011610.busancidc-html.com",
"https://010201.busancidc-html.com",
"https://010202.busancidc-html.com",
"https://010203.busancidc-html.com",
"https://010204.busancidc-html.com",
"https://010205.busancidc-html.com",
"https://010206.busancidc-html.com",
"https://010207.busancidc-html.com",
"https://010208.busancidc-html.com",
"https://010209.busancidc-html.com",
"https://010210.busancidc-html.com",
"https://012101.busancidc-html.com",
"https://012102.busancidc-html.com",
"https://012103.busancidc-html.com",
"https://012104.busancidc-html.com",
"https://012105.busancidc-html.com",
"https://012106.busancidc-html.com",
"https://012107.busancidc-html.com",
"https://012108.busancidc-html.com",
"https://012109.busancidc-html.com",
"https://012110.busancidc-html.com",
"https://011801.busancidc-html.com",
"https://011802.busancidc-html.com",
"https://011803.busancidc-html.com",
"https://011804.busancidc-html.com",
"https://011805.busancidc-html.com",
"https://011806.busancidc-html.com",
"https://011807.busancidc-html.com",
"https://011808.busancidc-html.com",
"https://011809.busancidc-html.com",
"https://011810.busancidc-html.com",
"https://011901.busancidc-html.com",
"https://011902.busancidc-html.com",
"https://011903.busancidc-html.com",
"https://011904.busancidc-html.com",
"https://011905.busancidc-html.com",
"https://011906.busancidc-html.com",
"https://011907.busancidc-html.com",
"https://011908.busancidc-html.com",
"https://011909.busancidc-html.com",
"https://011910.busancidc-html.com",
"https://010501.busancidc-html.com",
"https://010502.busancidc-html.com",
"https://010503.busancidc-html.com",
"https://010504.busancidc-html.com",
"https://010505.busancidc-html.com",
"https://010506.busancidc-html.com",
"https://010507.busancidc-html.com",
"https://010508.busancidc-html.com",
"https://010509.busancidc-html.com",
"https://010510.busancidc-html.com",
"https://011401.busancidc-html.com",
"https://011402.busancidc-html.com",
"https://011403.busancidc-html.com",
"https://011404.busancidc-html.com",
"https://011405.busancidc-html.com",
"https://011406.busancidc-html.com",
"https://011407.busancidc-html.com",
"https://011408.busancidc-html.com",
"https://011409.busancidc-html.com",
"https://011410.busancidc-html.com",
"https://011301.busancidc-html.com",
"https://011302.busancidc-html.com",
"https://011303.busancidc-html.com",
"https://011304.busancidc-html.com",
"https://011305.busancidc-html.com",
"https://011306.busancidc-html.com",
"https://011307.busancidc-html.com",
"https://011308.busancidc-html.com",
"https://011309.busancidc-html.com",
"https://011310.busancidc-html.com",
"https://021001.busancidc-html.com",
"https://021002.busancidc-html.com",
"https://021003.busancidc-html.com",
"https://021004.busancidc-html.com",
"https://021005.busancidc-html.com",
"https://021006.busancidc-html.com",
"https://021007.busancidc-html.com",
"https://021008.busancidc-html.com",
"https://021009.busancidc-html.com",
"https://021010.busancidc-html.com",
"https://021601.busancidc-html.com",
"https://021602.busancidc-html.com",
"https://021603.busancidc-html.com",
"https://021604.busancidc-html.com",
"https://021605.busancidc-html.com",
"https://021606.busancidc-html.com",
"https://021607.busancidc-html.com",
"https://021608.busancidc-html.com",
"https://021609.busancidc-html.com",
"https://021610.busancidc-html.com",
"https://022201.busancidc-html.com",
"https://022202.busancidc-html.com",
"https://022203.busancidc-html.com",
"https://022204.busancidc-html.com",
"https://022205.busancidc-html.com",
"https://022206.busancidc-html.com",
"https://022207.busancidc-html.com",
"https://022208.busancidc-html.com",
"https://022209.busancidc-html.com",
"https://022210.busancidc-html.com",
"https://020101.busancidc-html.com",
"https://020102.busancidc-html.com",
"https://202103.busancidc-html.com",
"https://020104.busancidc-html.com",
"https://020105.busancidc-html.com",
"https://020106.busancidc-html.com",
"https://020107.busancidc-html.com",
"https://020108.busancidc-html.com",
"https://020109.busancidc-html.com",
"https://020110.busancidc-html.com",
"https://022001.busancidc-html.com",
"https://022002.busancidc-html.com",
"https://022003.busancidc-html.com",
"https://022004.busancidc-html.com",
"https://022005.busancidc-html.com",
"https://022006.busancidc-html.com",
"https://022007.busancidc-html.com",
"https://022008.busancidc-html.com",
"https://022009.busancidc-html.com",
"https://022010.busancidc-html.com",
"https://021101.busancidc-html.com",
"https://021102.busancidc-html.com",
"https://021103.busancidc-html.com",
"https://021104.busancidc-html.com",
"https://021105.busancidc-html.com",
"https://021106.busancidc-html.com",
"https://021107.busancidc-html.com",
"https://021108.busancidc-html.com",
"https://021109.busancidc-html.com",
"https://021110.busancidc-html.com",
"https://021901.busancidc-html.com",
"https://021902.busancidc-html.com",
"https://021903.busancidc-html.com",
"https://021904.busancidc-html.com",
"https://021905.busancidc-html.com",
"https://021906.busancidc-html.com",
"https://021907.busancidc-html.com",
"https://021908.busancidc-html.com",
"https://021909.busancidc-html.com",
"https://021910.busancidc-html.com",
"https://020801.busancidc-html.com",
"https://020802.busancidc-html.com",
"https://020803.busancidc-html.com",
"https://020804.busancidc-html.com",
"https://020805.busancidc-html.com",
"https://020806.busancidc-html.com",
"https://020807.busancidc-html.com",
"https://020808.busancidc-html.com",
"https://020809.busancidc-html.com",
"https://020810.busancidc-html.com",
"https://020301.busancidc-html.com",
"https://020302.busancidc-html.com",
"https://020303.busancidc-html.com",
"https://020304.busancidc-html.com",
"https://020305.busancidc-html.com",
"https://020306.busancidc-html.com",
"https://020307.busancidc-html.com",
"https://020308.busancidc-html.com",
"https://020309.busancidc-html.com",
"https://020310.busancidc-html.com",
"https://021501.busancidc-html.com",
"https://021502.busancidc-html.com",
"https://021503.busancidc-html.com",
"https://021504.busancidc-html.com",
"https://021505.busancidc-html.com",
"https://021506.busancidc-html.com",
"https://021507.busancidc-html.com",
"https://021508.busancidc-html.com",
"https://021509.busancidc-html.com",
"https://021510.busancidc-html.com",
"https://021201.busancidc-html.com",
"https://021202.busancidc-html.com",
"https://021203.busancidc-html.com",
"https://021204.busancidc-html.com",
"https://021205.busancidc-html.com",
"https://021206.busancidc-html.com",
"https://021207.busancidc-html.com",
"https://021208.busancidc-html.com",
"https://021209.busancidc-html.com",
"https://021210.busancidc-html.com",
"https://020201.busancidc-html.com",
"https://020202.busancidc-html.com",
"https://020203.busancidc-html.com",
"https://020204.busancidc-html.com",
"https://020205.busancidc-html.com",
"https://020206.busancidc-html.com",
"https://020207.busancidc-html.com",
"https://020208.busancidc-html.com",
"https://020209.busancidc-html.com",
"https://020210.busancidc-html.com",
"https://020901.busancidc-html.com",
"https://020902.busancidc-html.com",
"https://020903.busancidc-html.com",
"https://020904.busancidc-html.com",
"https://020905.busancidc-html.com",
"https://020906.busancidc-html.com",
"https://020907.busancidc-html.com",
"https://020908.busancidc-html.com",
"https://020909.busancidc-html.com",
"https://020910.busancidc-html.com",
"https://020601.busancidc-html.com",
"https://020602.busancidc-html.com",
"https://020603.busancidc-html.com",
"https://020604.busancidc-html.com",
"https://020605.busancidc-html.com",
"https://020606.busancidc-html.com",
"https://020607.busancidc-html.com",
"https://020608.busancidc-html.com",
"https://020609.busancidc-html.com",
"https://020610.busancidc-html.com",
"https://022101.busancidc-html.com",
"https://022102.busancidc-html.com",
"https://022103.busancidc-html.com",
"https://022104.busancidc-html.com",
"https://022105.busancidc-html.com",
"https://022106.busancidc-html.com",
"https://022107.busancidc-html.com",
"https://022108.busancidc-html.com",
"https://022109.busancidc-html.com",
"https://022110.busancidc-html.com",
"https://020501.busancidc-html.com",
"https://020502.busancidc-html.com",
"https://020503.busancidc-html.com",
"https://020504.busancidc-html.com",
"https://020505.busancidc-html.com",
"https://020506.busancidc-html.com",
"https://020507.busancidc-html.com",
"https://020508.busancidc-html.com",
"https://020509.busancidc-html.com",
"https://020510.busancidc-html.com",
"https://020701.busancidc-html.com",
"https://020702.busancidc-html.com",
"https://020703.busancidc-html.com",
"https://020704.busancidc-html.com",
"https://020705.busancidc-html.com",
"https://020706.busancidc-html.com",
"https://020707.busancidc-html.com",
"https://020708.busancidc-html.com",
"https://020709.busancidc-html.com",
"https://020710.busancidc-html.com",
"https://021701.busancidc-html.com",
"https://021702.busancidc-html.com",
"https://021703.busancidc-html.com",
"https://021704.busancidc-html.com",
"https://021705.busancidc-html.com",
"https://021706.busancidc-html.com",
"https://021707.busancidc-html.com",
"https://021708.busancidc-html.com",
"https://021709.busancidc-html.com",
"https://021710.busancidc-html.com",
"https://021401.busancidc-html.com",
"https://021402.busancidc-html.com",
"https://021403.busancidc-html.com",
"https://021404.busancidc-html.com",
"https://021405.busancidc-html.com",
"https://021406.busancidc-html.com",
"https://021407.busancidc-html.com",
"https://021408.busancidc-html.com",
"https://021409.busancidc-html.com",
"https://021410.busancidc-html.com",
"https://020301.busancidc-html.com",
"https://021302.busancidc-html.com",
"https://021303.busancidc-html.com",
"https://021304.busancidc-html.com",
"https://021305.busancidc-html.com",
"https://021306.busancidc-html.com",
"https://021307.busancidc-html.com",
"https://021308.busancidc-html.com",
"https://021309.busancidc-html.com",
"https://021310.busancidc-html.com",
"https://021801.busancidc-html.com",
"https://021802.busancidc-html.com",
"https://021803.busancidc-html.com",
"https://021804.busancidc-html.com",
"https://021805.busancidc-html.com",
"https://021806.busancidc-html.com",
"https://021807.busancidc-html.com",
"https://021808.busancidc-html.com",
"https://021809.busancidc-html.com",
"https://021810.busancidc-html.com",
"https://020401.busancidc-html.com",
"https://020402.busancidc-html.com",
"https://020403.busancidc-html.com",
"https://020404.busancidc-html.com",
"https://020405.busancidc-html.com",
"https://020406.busancidc-html.com",
"https://020407.busancidc-html.com",
"https://020408.busancidc-html.com",
"https://020409.busancidc-html.com",
"https://020410.busancidc-html.com",
"https://030201.busancidc-html.com",
"https://030202.busancidc-html.com",
"https://030203.busancidc-html.com",
"https://030204.busancidc-html.com",
"https://030205.busancidc-html.com",
"https://030206.busancidc-html.com",
"https://030207.busancidc-html.com",
"https://030208.busancidc-html.com",
"https://030209.busancidc-html.com",
"https://030210.busancidc-html.com",
"https://030401.busancidc-html.com",
"https://030402.busancidc-html.com",
"https://030403.busancidc-html.com",
"https://030404.busancidc-html.com",
"https://030405.busancidc-html.com",
"https://030406.busancidc-html.com",
"https://030407.busancidc-html.com",
"https://030408.busancidc-html.com",
"https://030409.busancidc-html.com",
"https://030410.busancidc-html.com",
"https://031301.busancidc-html.com",
"https://031302.busancidc-html.com",
"https://031303.busancidc-html.com",
"https://031304.busancidc-html.com",
"https://031305.busancidc-html.com",
"https://031306.busancidc-html.com",
"https://031307.busancidc-html.com",
"https://031308.busancidc-html.com",
"https://031309.busancidc-html.com",
"https://031310.busancidc-html.com",
"https://031801.busancidc-html.com",
"https://031802.busancidc-html.com",
"https://031803.busancidc-html.com",
"https://031804.busancidc-html.com",
"https://031805.busancidc-html.com",
"https://031806.busancidc-html.com",
"https://031807.busancidc-html.com",
"https://031808.busancidc-html.com",
"https://031809.busancidc-html.com",
"https://031810.busancidc-html.com",
"https://032101.busancidc-html.com",
"https://032102.busancidc-html.com",
"https://032103.busancidc-html.com",
"https://032104.busancidc-html.com",
"https://032105.busancidc-html.com",
"https://032106.busancidc-html.com",
"https://032107.busancidc-html.com",
"https://032108.busancidc-html.com",
"https://032109.busancidc-html.com",
"https://032110.busancidc-html.com",
"https://030601.busancidc-html.com",
"https://030602.busancidc-html.com",
"https://030603.busancidc-html.com",
"https://030604.busancidc-html.com",
"https://030605.busancidc-html.com",
"https://030606.busancidc-html.com",
"https://030607.busancidc-html.com",
"https://030608.busancidc-html.com",
"https://030609.busancidc-html.com",
"https://030610.busancidc-html.com",
"https://031101.busancidc-html.com",
"https://031102.busancidc-html.com",
"https://031103.busancidc-html.com",
"https://031104.busancidc-html.com",
"https://031105.busancidc-html.com",
"https://031106.busancidc-html.com",
"https://031107.busancidc-html.com",
"https://031108.busancidc-html.com",
"https://031109.busancidc-html.com",
"https://031110.busancidc-html.com",
"https://030501.busancidc-html.com",
"https://030502.busancidc-html.com",
"https://030503.busancidc-html.com",
"https://030504.busancidc-html.com",
"https://030505.busancidc-html.com",
"https://030506.busancidc-html.com",
"https://030507.busancidc-html.com",
"https://030508.busancidc-html.com",
"https://030509.busancidc-html.com",
"https://030510.busancidc-html.com",
"https://030901.busancidc-html.com",
"https://030902.busancidc-html.com",
"https://030903.busancidc-html.com",
"https://030904.busancidc-html.com",
"https://030905.busancidc-html.com",
"https://030906.busancidc-html.com",
"https://030907.busancidc-html.com",
"https://030908.busancidc-html.com",
"https://030909.busancidc-html.com",
"https://030910.busancidc-html.com",
"https://030801.busancidc-html.com",
"https://030802.busancidc-html.com",
"https://030803.busancidc-html.com",
"https://030804.busancidc-html.com",
"https://030805.busancidc-html.com",
"https://030806.busancidc-html.com",
"https://030807.busancidc-html.com",
"https://030808.busancidc-html.com",
"https://030809.busancidc-html.com",
"https://030810.busancidc-html.com",
"https://031201.busancidc-html.com",
"https://031202.busancidc-html.com",
"https://031203.busancidc-html.com",
"https://031204.busancidc-html.com",
"https://031205.busancidc-html.com",
"https://031206.busancidc-html.com",
"https://031207.busancidc-html.com",
"https://031208.busancidc-html.com",
"https://031209.busancidc-html.com",
"https://031210.busancidc-html.com",
"https://030701.busancidc-html.com",
"https://030702.busancidc-html.com",
"https://030703.busancidc-html.com",
"https://030704.busancidc-html.com",
"https://030705.busancidc-html.com",
"https://030706.busancidc-html.com",
"https://030707.busancidc-html.com",
"https://030708.busancidc-html.com",
"https://030709.busancidc-html.com",
"https://030710.busancidc-html.com",
"https://031401.busancidc-html.com",
"https://031402.busancidc-html.com",
"https://031403.busancidc-html.com",
"https://031404.busancidc-html.com",
"https://031405.busancidc-html.com",
"https://031406.busancidc-html.com",
"https://031407.busancidc-html.com",
"https://031408.busancidc-html.com",
"https://031409.busancidc-html.com",
"https://031410.busancidc-html.com",
"https://032701.busancidc-html.com",
"https://032702.busancidc-html.com",
"https://032703.busancidc-html.com",
"https://032704.busancidc-html.com",
"https://032705.busancidc-html.com",
"https://032706.busancidc-html.com",
"https://032707.busancidc-html.com",
"https://032708.busancidc-html.com",
"https://032709.busancidc-html.com",
"https://032710.busancidc-html.com",
"https://032001.busancidc-html.com",
"https://032002.busancidc-html.com",
"https://032003.busancidc-html.com",
"https://032004.busancidc-html.com",
"https://032005.busancidc-html.com",
"https://032006.busancidc-html.com",
"https://032007.busancidc-html.com",
"https://032008.busancidc-html.com",
"https://032009.busancidc-html.com",
"https://032010.busancidc-html.com",
"https://032301.busancidc-html.com",
"https://032302.busancidc-html.com",
"https://032303.busancidc-html.com",
"https://032304.busancidc-html.com",
"https://032305.busancidc-html.com",
"https://032306.busancidc-html.com",
"https://032307.busancidc-html.com",
"https://032308.busancidc-html.com",
"https://032309.busancidc-html.com",
"https://032310.busancidc-html.com",
"https://030301.busancidc-html.com",
"https://030302.busancidc-html.com",
"https://030303.busancidc-html.com",
"https://030304.busancidc-html.com",
"https://030305.busancidc-html.com",
"https://030306.busancidc-html.com",
"https://030307.busancidc-html.com",
"https://030308.busancidc-html.com",
"https://030309.busancidc-html.com",
"https://030310.busancidc-html.com",
"https://032501.busancidc-html.com",
"https://032502.busancidc-html.com",
"https://032503.busancidc-html.com",
"https://032504.busancidc-html.com",
"https://032505.busancidc-html.com",
"https://032506.busancidc-html.com",
"https://032507.busancidc-html.com",
"https://032508.busancidc-html.com",
"https://032509.busancidc-html.com",
"https://032510.busancidc-html.com",
"https://032601.busancidc-html.com",
"https://032602.busancidc-html.com",
"https://032603.busancidc-html.com",
"https://032604.busancidc-html.com",
"https://032605.busancidc-html.com",
"https://032606.busancidc-html.com",
"https://032607.busancidc-html.com",
"https://032608.busancidc-html.com",
"https://032609.busancidc-html.com",
"https://032610.busancidc-html.com",
"https://032201.busancidc-html.com",
"https://032202.busancidc-html.com",
"https://032203.busancidc-html.com",
"https://032204.busancidc-html.com",
"https://032205.busancidc-html.com",
"https://032206.busancidc-html.com",
"https://032207.busancidc-html.com",
"https://032208.busancidc-html.com",
"https://032209.busancidc-html.com",
"https://032210.busancidc-html.com",
"https://031001.busancidc-html.com",
"https://031002.busancidc-html.com",
"https://031003.busancidc-html.com",
"https://031004.busancidc-html.com",
"https://031005.busancidc-html.com",
"https://031006.busancidc-html.com",
"https://031007.busancidc-html.com",
"https://031008.busancidc-html.com",
"https://031009.busancidc-html.com",
"https://031010.busancidc-html.com",
"https://032401.busancidc-html.com",
"https://032402.busancidc-html.com",
"https://032403.busancidc-html.com",
"https://032404.busancidc-html.com",
"https://032405.busancidc-html.com",
"https://032406.busancidc-html.com",
"https://032407.busancidc-html.com",
"https://032408.busancidc-html.com",
"https://032409.busancidc-html.com",
"https://032410.busancidc-html.com",
"https://031901.busancidc-html.com",
"https://031902.busancidc-html.com",
"https://031903.busancidc-html.com",
"https://031904.busancidc-html.com",
"https://031905.busancidc-html.com",
"https://031906.busancidc-html.com",
"https://031907.busancidc-html.com",
"https://031908.busancidc-html.com",
"https://031909.busancidc-html.com",
"https://031910.busancidc-html.com",
"https://031601.busancidc-html.com",
"https://031602.busancidc-html.com",
"https://031603.busancidc-html.com",
"https://031604.busancidc-html.com",
"https://031605.busancidc-html.com",
"https://031606.busancidc-html.com",
"https://031607.busancidc-html.com",
"https://031608.busancidc-html.com",
"https://031609.busancidc-html.com",
"https://031610.busancidc-html.com",
"https://030101.busancidc-html.com",
"https://030102.busancidc-html.com",
"https://030103.busancidc-html.com",
"https://030104.busancidc-html.com",
"https://030105.busancidc-html.com",
"https://030106.busancidc-html.com",
"https://030107.busancidc-html.com",
"https://030108.busancidc-html.com",
"https://030109.busancidc-html.com",
"https://030110.busancidc-html.com",
"https://031701.busancidc-html.com",
"https://031702.busancidc-html.com",
"https://031703.busancidc-html.com",
"https://031704.busancidc-html.com",
"https://031705.busancidc-html.com",
"https://031706.busancidc-html.com",
"https://031707.busancidc-html.com",
"https://031708.busancidc-html.com",
"https://031709.busancidc-html.com",
"https://031710.busancidc-html.com",
"https://031501.busancidc-html.com",
"https://031502.busancidc-html.com",
"https://031503.busancidc-html.com",
"https://031504.busancidc-html.com",
"https://031505.busancidc-html.com",
"https://031506.busancidc-html.com",
"https://031507.busancidc-html.com",
"https://031508.busancidc-html.com",
"https://031509.busancidc-html.com",
"https://031510.busancidc-html.com",
"https://032801.busancidc-html.com",
"https://032802.busancidc-html.com",
"https://032803.busancidc-html.com",
"https://032804.busancidc-html.com",
"https://032805.busancidc-html.com",
"https://032806.busancidc-html.com",
"https://032807.busancidc-html.com",
"https://032808.busancidc-html.com",
"https://032809.busancidc-html.com",
"https://032810.busancidc-html.com",
"https://040501.busancidc-html.com",
"https://040502.busancidc-html.com",
"https://040503.busancidc-html.com",
"https://040504.busancidc-html.com",
"https://040505.busancidc-html.com",
"https://040506.busancidc-html.com",
"https://040507.busancidc-html.com",
"https://040508.busancidc-html.com",
"https://040509.busancidc-html.com",
"https://040510.busancidc-html.com",
"https://045501.busancidc-html.com",
"https://045502.busancidc-html.com",
"https://045503.busancidc-html.com",
"https://045504.busancidc-html.com",
"https://045505.busancidc-html.com",
"https://045506.busancidc-html.com",
"https://045507.busancidc-html.com",
"https://045508.busancidc-html.com",
"https://045509.busancidc-html.com",
"https://045510.busancidc-html.com",
"https://043001.busancidc-html.com",
"https://043002.busancidc-html.com",
"https://043003.busancidc-html.com",
"https://043004.busancidc-html.com",
"https://043005.busancidc-html.com",
"https://043006.busancidc-html.com",
"https://043007.busancidc-html.com",
"https://043008.busancidc-html.com",
"https://043009.busancidc-html.com",
"https://043010.busancidc-html.com",
"https://043901.busancidc-html.com",
"https://043902.busancidc-html.com",
"https://043903.busancidc-html.com",
"https://043904.busancidc-html.com",
"https://043905.busancidc-html.com",
"https://043906.busancidc-html.com",
"https://043907.busancidc-html.com",
"https://043908.busancidc-html.com",
"https://043909.busancidc-html.com",
"https://043910.busancidc-html.com",
"https://043301.busancidc-html.com",
"https://043302.busancidc-html.com",
"https://043303.busancidc-html.com",
"https://043304.busancidc-html.com",
"https://043305.busancidc-html.com",
"https://043306.busancidc-html.com",
"https://043307.busancidc-html.com",
"https://043308.busancidc-html.com",
"https://043309.busancidc-html.com",
"https://043310.busancidc-html.com",
"https://041601.busancidc-html.com",
"https://041602.busancidc-html.com",
"https://041603.busancidc-html.com",
"https://041604.busancidc-html.com",
"https://041605.busancidc-html.com",
"https://041606.busancidc-html.com",
"https://041607.busancidc-html.com",
"https://041608.busancidc-html.com",
"https://041609.busancidc-html.com",
"https://041610.busancidc-html.com",
"https://041701.busancidc-html.com",
"https://041702.busancidc-html.com",
"https://041703.busancidc-html.com",
"https://041704.busancidc-html.com",
"https://041705.busancidc-html.com",
"https://041706.busancidc-html.com",
"https://041707.busancidc-html.com",
"https://041708.busancidc-html.com",
"https://041709.busancidc-html.com",
"https://041710.busancidc-html.com",
"https://043601.busancidc-html.com",
"https://043602.busancidc-html.com",
"https://043603.busancidc-html.com",
"https://043604.busancidc-html.com",
"https://043605.busancidc-html.com",
"https://043606.busancidc-html.com",
"https://043607.busancidc-html.com",
"https://043608.busancidc-html.com",
"https://043609.busancidc-html.com",
"https://043610.busancidc-html.com",
"https://044401.busancidc-html.com",
"https://044402.busancidc-html.com",
"https://044403.busancidc-html.com",
"https://044404.busancidc-html.com",
"https://044405.busancidc-html.com",
"https://044406.busancidc-html.com",
"https://044407.busancidc-html.com",
"https://044408.busancidc-html.com",
"https://044409.busancidc-html.com",
"https://044410.busancidc-html.com",
"https://045001.busancidc-html.com",
"https://045002.busancidc-html.com",
"https://045003.busancidc-html.com",
"https://045004.busancidc-html.com",
"https://045005.busancidc-html.com",
"https://045006.busancidc-html.com",
"https://045007.busancidc-html.com",
"https://045008.busancidc-html.com",
"https://045009.busancidc-html.com",
"https://045010.busancidc-html.com",
"https://042901.busancidc-html.com",
"https://042902.busancidc-html.com",
"https://042903.busancidc-html.com",
"https://042904.busancidc-html.com",
"https://042905.busancidc-html.com",
"https://042906.busancidc-html.com",
"https://042907.busancidc-html.com",
"https://042908.busancidc-html.com",
"https://042909.busancidc-html.com",
"https://042910.busancidc-html.com",
"https://044701.busancidc-html.com",
"https://044702.busancidc-html.com",
"https://044703.busancidc-html.com",
"https://044704.busancidc-html.com",
"https://044705.busancidc-html.com",
"https://044706.busancidc-html.com",
"https://044707.busancidc-html.com",
"https://044708.busancidc-html.com",
"https://044709.busancidc-html.com",
"https://044710.busancidc-html.com",
"https://046001.busancidc-html.com",
"https://046002.busancidc-html.com",
"https://046003.busancidc-html.com",
"https://046004.busancidc-html.com",
"https://046005.busancidc-html.com",
"https://046006.busancidc-html.com",
"https://046007.busancidc-html.com",
"https://046008.busancidc-html.com",
"https://046009.busancidc-html.com",
"https://046010.busancidc-html.com",
"https://041501.busancidc-html.com",
"https://041502.busancidc-html.com",
"https://041503.busancidc-html.com",
"https://041504.busancidc-html.com",
"https://041505.busancidc-html.com",
"https://041506.busancidc-html.com",
"https://041507.busancidc-html.com",
"https://041508.busancidc-html.com",
"https://041509.busancidc-html.com",
"https://041510.busancidc-html.com",
"https://045101.busancidc-html.com",
"https://045102.busancidc-html.com",
"https://045103.busancidc-html.com",
"https://045104.busancidc-html.com",
"https://045105.busancidc-html.com",
"https://045106.busancidc-html.com",
"https://045107.busancidc-html.com",
"https://045108.busancidc-html.com",
"https://045109.busancidc-html.com",
"https://045110.busancidc-html.com",
"https://042701.busancidc-html.com",
"https://042702.busancidc-html.com",
"https://042703.busancidc-html.com",
"https://042704.busancidc-html.com",
"https://042705.busancidc-html.com",
"https://042706.busancidc-html.com",
"https://042707.busancidc-html.com",
"https://042708.busancidc-html.com",
"https://042709.busancidc-html.com",
"https://042710.busancidc-html.com",
"https://041401.busancidc-html.com",
"https://041402.busancidc-html.com",
"https://041403.busancidc-html.com",
"https://041404.busancidc-html.com",
"https://041405.busancidc-html.com",
"https://041406.busancidc-html.com",
"https://041407.busancidc-html.com",
"https://041408.busancidc-html.com",
"https://041409.busancidc-html.com",
"https://041410.busancidc-html.com",
"https://044501.busancidc-html.com",
"https://044502.busancidc-html.com",
"https://044503.busancidc-html.com",
"https://044504.busancidc-html.com",
"https://044505.busancidc-html.com",
"https://044506.busancidc-html.com",
"https://044507.busancidc-html.com",
"https://044508.busancidc-html.com",
"https://044509.busancidc-html.com",
"https://044510.busancidc-html.com",
"https://044101.busancidc-html.com",
"https://044102.busancidc-html.com",
"https://044103.busancidc-html.com",
"https://044104.busancidc-html.com",
"https://044105.busancidc-html.com",
"https://044106.busancidc-html.com",
"https://044107.busancidc-html.com",
"https://044108.busancidc-html.com",
"https://044109.busancidc-html.com",
"https://044110.busancidc-html.com",
"https://044601.busancidc-html.com",
"https://044602.busancidc-html.com",
"https://044603.busancidc-html.com",
"https://044604.busancidc-html.com",
"https://044605.busancidc-html.com",
"https://044606.busancidc-html.com",
"https://044607.busancidc-html.com",
"https://044608.busancidc-html.com",
"https://044609.busancidc-html.com",
"https://044610.busancidc-html.com",
"https://046201.busancidc-html.com",
"https://046202.busancidc-html.com",
"https://046203.busancidc-html.com",
"https://046205.busancidc-html.com",
"https://046206.busancidc-html.com",
"https://046207.busancidc-html.com",
"https://046208.busancidc-html.com",
"https://046209.busancidc-html.com",
"https://046210.busancidc-html.com",
"https://045701.busancidc-html.com",
"https://045702.busancidc-html.com",
"https://045703.busancidc-html.com",
"https://045704.busancidc-html.com",
"https://045705.busancidc-html.com",
"https://045706.busancidc-html.com",
"https://045707.busancidc-html.com",
"https://045708.busancidc-html.com",
"https://045709.busancidc-html.com",
"https://045710.busancidc-html.com",
"https://043401.busancidc-html.com",
"https://043402.busancidc-html.com",
"https://043403.busancidc-html.com",
"https://043404.busancidc-html.com",
"https://043405.busancidc-html.com",
"https://043406.busancidc-html.com",
"https://043407.busancidc-html.com",
"https://043408.busancidc-html.com",
"https://043409.busancidc-html.com",
"https://043410.busancidc-html.com",
"https://041901.busancidc-html.com",
"https://041902.busancidc-html.com",
"https://041903.busancidc-html.com",
"https://041904.busancidc-html.com",
"https://041905.busancidc-html.com",
"https://041906.busancidc-html.com",
"https://041907.busancidc-html.com",
"https://041908.busancidc-html.com",
"https://041909.busancidc-html.com",
"https://041910.busancidc-html.com",
"https://045401.busancidc-html.com",
"https://045402.busancidc-html.com",
"https://045403.busancidc-html.com",
"https://045404.busancidc-html.com",
"https://045405.busancidc-html.com",
"https://045406.busancidc-html.com",
"https://045407.busancidc-html.com",
"https://045408.busancidc-html.com",
"https://045409.busancidc-html.com",
"https://045410.busancidc-html.com",
"https://041201.busancidc-html.com",
"https://041202.busancidc-html.com",
"https://041203.busancidc-html.com",
"https://041204.busancidc-html.com",
"https://041205.busancidc-html.com",
"https://041206.busancidc-html.com",
"https://041207.busancidc-html.com",
"https://041208.busancidc-html.com",
"https://041209.busancidc-html.com",
"https://041210.busancidc-html.com",
"https://040801.busancidc-html.com",
"https://040802.busancidc-html.com",
"https://040803.busancidc-html.com",
"https://040804.busancidc-html.com",
"https://040805.busancidc-html.com",
"https://040806.busancidc-html.com",
"https://040807.busancidc-html.com",
"https://040808.busancidc-html.com",
"https://040809.busancidc-html.com",
"https://040810.busancidc-html.com",
"https://044001.busancidc-html.com",
"https://044002.busancidc-html.com",
"https://044003.busancidc-html.com",
"https://044004.busancidc-html.com",
"https://044005.busancidc-html.com",
"https://044006.busancidc-html.com",
"https://044007.busancidc-html.com",
"https://044008.busancidc-html.com",
"https://044009.busancidc-html.com",
"https://044010.busancidc-html.com",
"https://043101.busancidc-html.com",
"https://043102.busancidc-html.com",
"https://043103.busancidc-html.com",
"https://043104.busancidc-html.com",
"https://043105.busancidc-html.com",
"https://043106.busancidc-html.com",
"https://043107.busancidc-html.com",
"https://043108.busancidc-html.com",
"https://043109.busancidc-html.com",
"https://043110.busancidc-html.com",
"https://045201.busancidc-html.com",
"https://045202.busancidc-html.com",
"https://045203.busancidc-html.com",
"https://045204.busancidc-html.com",
"https://045205.busancidc-html.com",
"https://045206.busancidc-html.com",
"https://045207.busancidc-html.com",
"https://045208.busancidc-html.com",
"https://045209.busancidc-html.com",
"https://045210.busancidc-html.com",
"https://045601.busancidc-html.com",
"https://045602.busancidc-html.com",
"https://045603.busancidc-html.com",
"https://045604.busancidc-html.com",
"https://045605.busancidc-html.com",
"https://045606.busancidc-html.com",
"https://045607.busancidc-html.com",
"https://045608.busancidc-html.com",
"https://045609.busancidc-html.com",
"https://045610.busancidc-html.com",
"https://046101.busancidc-html.com",
"https://046102.busancidc-html.com",
"https://046103.busancidc-html.com",
"https://046104.busancidc-html.com",
"https://046105.busancidc-html.com",
"https://046106.busancidc-html.com",
"https://046107.busancidc-html.com",
"https://046108.busancidc-html.com",
"https://046109.busancidc-html.com",
"https://046110.busancidc-html.com",
"https://041101.busancidc-html.com",
"https://041102.busancidc-html.com",
"https://041103.busancidc-html.com",
"https://041104.busancidc-html.com",
"https://041105.busancidc-html.com",
"https://041106.busancidc-html.com",
"https://041107.busancidc-html.com",
"https://041108.busancidc-html.com",
"https://041109.busancidc-html.com",
"https://041110.busancidc-html.com",
"https://042801.busancidc-html.com",
"https://042802.busancidc-html.com",
"https://042803.busancidc-html.com",
"https://042804.busancidc-html.com",
"https://042805.busancidc-html.com",
"https://042806.busancidc-html.com",
"https://042807.busancidc-html.com",
"https://042808.busancidc-html.com",
"https://042809.busancidc-html.com",
"https://042810.busancidc-html.com",
"https://040401.busancidc-html.com",
"https://040402.busancidc-html.com",
"https://040403.busancidc-html.com",
"https://040404.busancidc-html.com",
"https://040405.busancidc-html.com",
"https://040406.busancidc-html.com",
"https://040407.busancidc-html.com",
"https://040408.busancidc-html.com",
"https://040409.busancidc-html.com",
"https://040410.busancidc-html.com",
"https://043801.busancidc-html.com",
"https://043802.busancidc-html.com",
"https://043803.busancidc-html.com",
"https://043804.busancidc-html.com",
"https://043805.busancidc-html.com",
"https://043806.busancidc-html.com",
"https://043807.busancidc-html.com",
"https://043808.busancidc-html.com",
"https://043809.busancidc-html.com",
"https://043810.busancidc-html.com",
"https://043501.busancidc-html.com",
"https://043502.busancidc-html.com",
"https://043503.busancidc-html.com",
"https://043504.busancidc-html.com",
"https://043505.busancidc-html.com",
"https://043506.busancidc-html.com",
"https://043507.busancidc-html.com",
"https://043508.busancidc-html.com",
"https://043509.busancidc-html.com",
"https://043510.busancidc-html.com",
"https://040101.busancidc-html.com",
"https://040102.busancidc-html.com",
"https://040103.busancidc-html.com",
"https://040104.busancidc-html.com",
"https://040105.busancidc-html.com",
"https://040106.busancidc-html.com",
"https://040107.busancidc-html.com",
"https://040108.busancidc-html.com",
"https://040109.busancidc-html.com",
"https://040110.busancidc-html.com",
"https://040901.busancidc-html.com",
"https://040902.busancidc-html.com",
"https://040903.busancidc-html.com",
"https://040904.busancidc-html.com",
"https://040905.busancidc-html.com",
"https://040906.busancidc-html.com",
"https://040907.busancidc-html.com",
"https://040908.busancidc-html.com",
"https://040909.busancidc-html.com",
"https://040910.busancidc-html.com",
"https://043701.busancidc-html.com",
"https://043702.busancidc-html.com",
"https://043703.busancidc-html.com",
"https://043704.busancidc-html.com",
"https://043705.busancidc-html.com",
"https://043706.busancidc-html.com",
"https://043707.busancidc-html.com",
"https://043708.busancidc-html.com",
"https://043709.busancidc-html.com",
"https://043710.busancidc-html.com",
"https://041801.busancidc-html.com",
"https://041803.busancidc-html.com",
"https://041804.busancidc-html.com",
"https://041806.busancidc-html.com",
"https://041808.busancidc-html.com",
"https://041809.busancidc-html.com",
"https://041810.busancidc-html.com",
"https://043201.busancidc-html.com",
"https://043202.busancidc-html.com",
"https://043203.busancidc-html.com",
"https://043204.busancidc-html.com",
"https://043205.busancidc-html.com",
"https://043206.busancidc-html.com",
"https://043207.busancidc-html.com",
"https://043208.busancidc-html.com",
"https://043209.busancidc-html.com",
"https://043210.busancidc-html.com",
"https://042101.busancidc-html.com",
"https://042102.busancidc-html.com",
"https://042103.busancidc-html.com",
"https://042104.busancidc-html.com",
"https://042105.busancidc-html.com",
"https://042106.busancidc-html.com",
"https://042107.busancidc-html.com",
"https://042108.busancidc-html.com",
"https://042109.busancidc-html.com",
"https://042110.busancidc-html.com",
"https://042301.busancidc-html.com",
"https://042302.busancidc-html.com",
"https://042303.busancidc-html.com",
"https://042304.busancidc-html.com",
"https://042305.busancidc-html.com",
"https://042306.busancidc-html.com",
"https://042307.busancidc-html.com",
"https://042308.busancidc-html.com",
"https://042309.busancidc-html.com",
"https://042310.busancidc-html.com",
"https://042001.busancidc-html.com",
"https://042002.busancidc-html.com",
"https://042003.busancidc-html.com",
"https://042004.busancidc-html.com",
"https://042005.busancidc-html.com",
"https://042006.busancidc-html.com",
"https://042007.busancidc-html.com",
"https://042008.busancidc-html.com",
"https://042009.busancidc-html.com",
"https://042010.busancidc-html.com",
"https://042201.busancidc-html.com",
"https://042202.busancidc-html.com",
"https://042203.busancidc-html.com",
"https://042204.busancidc-html.com",
"https://042205.busancidc-html.com",
"https://042206.busancidc-html.com",
"https://042207.busancidc-html.com",
"https://042208.busancidc-html.com",
"https://042209.busancidc-html.com",
"https://042210.busancidc-html.com",
"https://040701.busancidc-html.com",
"https://040702.busancidc-html.com",
"https://040703.busancidc-html.com",
"https://040704.busancidc-html.com",
"https://040705.busancidc-html.com",
"https://040706.busancidc-html.com",
"https://040707.busancidc-html.com",
"https://040708.busancidc-html.com",
"https://040709.busancidc-html.com",
"https://040710.busancidc-html.com",
"https://045301.busancidc-html.com",
"https://045302.busancidc-html.com",
"https://045303.busancidc-html.com",
"https://045304.busancidc-html.com",
"https://045305.busancidc-html.com",
"https://045306.busancidc-html.com",
"https://045307.busancidc-html.com",
"https://045308.busancidc-html.com",
"https://045309.busancidc-html.com",
"https://045310.busancidc-html.com",
"https://041301.busancidc-html.com",
"https://041302.busancidc-html.com",
"https://041303.busancidc-html.com",
"https://041304.busancidc-html.com",
"https://041305.busancidc-html.com",
"https://041306.busancidc-html.com",
"https://041307.busancidc-html.com",
"https://041308.busancidc-html.com",
"https://041309.busancidc-html.com",
"https://041310.busancidc-html.com",
"https://042401.busancidc-html.com",
"https://042402.busancidc-html.com",
"https://042403.busancidc-html.com",
"https://042404.busancidc-html.com",
"https://042405.busancidc-html.com",
"https://042406.busancidc-html.com",
"https://042407.busancidc-html.com",
"https://042408.busancidc-html.com",
"https://042409.busancidc-html.com",
"https://042410.busancidc-html.com",
"https://046301.busancidc-html.com",
"https://046302.busancidc-html.com",
"https://046303.busancidc-html.com",
"https://046304.busancidc-html.com",
"https://046305.busancidc-html.com",
"https://046306.busancidc-html.com",
"https://046307.busancidc-html.com",
"https://046308.busancidc-html.com",
"https://046309.busancidc-html.com",
"https://046310.busancidc-html.com",
"https://041001.busancidc-html.com",
"https://041002.busancidc-html.com",
"https://041003.busancidc-html.com",
"https://041004.busancidc-html.com",
"https://041005.busancidc-html.com",
"https://041006.busancidc-html.com",
"https://041007.busancidc-html.com",
"https://041008.busancidc-html.com",
"https://041009.busancidc-html.com",
"https://041010.busancidc-html.com",
"https://044801.busancidc-html.com",
"https://044802.busancidc-html.com",
"https://044803.busancidc-html.com",
"https://044804.busancidc-html.com",
"https://044805.busancidc-html.com",
"https://044806.busancidc-html.com",
"https://044807.busancidc-html.com",
"https://044808.busancidc-html.com",
"https://044809.busancidc-html.com",
"https://044810.busancidc-html.com",
"https://042501.busancidc-html.com",
"https://042502.busancidc-html.com",
"https://042503.busancidc-html.com",
"https://042504.busancidc-html.com",
"https://042505.busancidc-html.com",
"https://042506.busancidc-html.com",
"https://042507.busancidc-html.com",
"https://042508.busancidc-html.com",
"https://042509.busancidc-html.com",
"https://042510.busancidc-html.com",
"https://045901.busancidc-html.com",
"https://045902.busancidc-html.com",
"https://045903.busancidc-html.com",
"https://045904.busancidc-html.com",
"https://045905.busancidc-html.com",
"https://045906.busancidc-html.com",
"https://045907.busancidc-html.com",
"https://045908.busancidc-html.com",
"https://045909.busancidc-html.com",
"https://045910.busancidc-html.com",
"https://044201.busancidc-html.com",
"https://044202.busancidc-html.com",
"https://044203.busancidc-html.com",
"https://044204.busancidc-html.com",
"https://044205.busancidc-html.com",
"https://044206.busancidc-html.com",
"https://044207.busancidc-html.com",
"https://044208.busancidc-html.com",
"https://044209.busancidc-html.com",
"https://044210.busancidc-html.com",
"https://040301.busancidc-html.com",
"https://040302.busancidc-html.com",
"https://040303.busancidc-html.com",
"https://040304.busancidc-html.com",
"https://040305.busancidc-html.com",
"https://040306.busancidc-html.com",
"https://040307.busancidc-html.com",
"https://040308.busancidc-html.com",
"https://040309.busancidc-html.com",
"https://040310.busancidc-html.com",
"https://040601.busancidc-html.com",
"https://040602.busancidc-html.com",
"https://040603.busancidc-html.com",
"https://040604.busancidc-html.com",
"https://040605.busancidc-html.com",
"https://040606.busancidc-html.com",
"https://040607.busancidc-html.com",
"https://040608.busancidc-html.com",
"https://040609.busancidc-html.com",
"https://040610.busancidc-html.com",
"https://045801.busancidc-html.com",
"https://045802.busancidc-html.com",
"https://045803.busancidc-html.com",
"https://045804.busancidc-html.com",
"https://045805.busancidc-html.com",
"https://045806.busancidc-html.com",
"https://045807.busancidc-html.com",
"https://045808.busancidc-html.com",
"https://045809.busancidc-html.com",
"https://045810.busancidc-html.com",
"https://044901.busancidc-html.com",
"https://044903.busancidc-html.com",
"https://044904.busancidc-html.com",
"https://044906.busancidc-html.com",
"https://044908.busancidc-html.com",
"https://044909.busancidc-html.com",
"https://044910.busancidc-html.com",
"https://044301.busancidc-html.com",
"https://044302.busancidc-html.com",
"https://044303.busancidc-html.com",
"https://044304.busancidc-html.com",
"https://044305.busancidc-html.com",
"https://044306.busancidc-html.com",
"https://044307.busancidc-html.com",
"https://044308.busancidc-html.com",
"https://044309.busancidc-html.com",
"https://044310.busancidc-html.com",
"https://042601.busancidc-html.com",
"https://042602.busancidc-html.com",
"https://042603.busancidc-html.com",
"https://042604.busancidc-html.com",
"https://042605.busancidc-html.com",
"https://042606.busancidc-html.com",
"https://042607.busancidc-html.com",
"https://042608.busancidc-html.com",
"https://042609.busancidc-html.com",
"https://042610.busancidc-html.com",
"https://040201.busancidc-html.com",
"https://040202.busancidc-html.com",
"https://040203.busancidc-html.com",
"https://040204.busancidc-html.com",
"https://040205.busancidc-html.com",
"https://040206.busancidc-html.com",
"https://040207.busancidc-html.com",
"https://040208.busancidc-html.com",
"https://040209.busancidc-html.com",
"https://040210.busancidc-html.com",
"https://000301.busancidc-html.com",
"https://000302.busancidc-html.com",
"https://000303.busancidc-html.com",
"https://000304.busancidc-html.com",
"https://000305.busancidc-html.com",
"https://000306.busancidc-html.com",
"https://000307.busancidc-html.com",
"https://000308.busancidc-html.com",
"https://000309.busancidc-html.com",
"https://000310.busancidc-html.com",
"https://000201.busancidc-html.com",
"https://000202.busancidc-html.com",
"https://000203.busancidc-html.com",
"https://000204.busancidc-html.com",
"https://000205.busancidc-html.com",
"https://000206.busancidc-html.com",
"https://000207.busancidc-html.com",
"https://000208.busancidc-html.com",
"https://000209.busancidc-html.com",
"https://000210.busancidc-html.com",
"https://000101.busancidc-html.com",
"https://000102.busancidc-html.com",
"https://000103.busancidc-html.com",
"https://000104.busancidc-html.com",
"https://000105.busancidc-html.com",
"https://000106.busancidc-html.com",
"https://000107.busancidc-html.com",
"https://000108.busancidc-html.com",
"https://000109.busancidc-html.com",
"https://000110.busancidc-html.com"
]


In [43]:
loader = WebBaseLoader(
    web_paths=urls,
    bs_kwargs=dict(),
    header_template={
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/102.0.0.0 Safari/537.36",
    },
)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1100, chunk_overlap=0)

docs = loader.load_and_split(text_splitter=text_splitter)

print(f'문서의 수 : {len(docs)}')
print(docs[0].metadata['source'])
print(docs[0].page_content)

문서의 수 : 1540
https://010601.busancidc-html.com
-남아메리카출혈열 질병개요 -구 분구 분내 용감염병 분류◦ 제1급 법정감염병◦ 생물테러감염병(고위험병원제 제4위험군, 생물안전밀폐등급 BL4)정의◦ 남아메리카에서 발생한 출혈열 바이러스 감염에 의한 급성 발열성 출혈성 질환*을 총칭함* 아르헨티나출혈열, 볼리비아출혈열, 베네수엘라출혈열, 브라질출혈열원인병원체◦ 남아메리카출혈열의 종류와 원인 바이러스 <Machupo 바이러스> - https://phil.cdc.gov/Details.aspx?pid=1869 -* 각 국가별로 다른 바이러스・매개 설치류에 의해 발생- 아르헨티나출혈열(Argentine Hemorrhagic Fever, AHF): Junín virus- 볼리비아출혈열(Bolivian Hemorrhagic Fever, BHF): Machupo virus- 베네수엘라출혈열(Venezuelan Hemorrhagic Fever, VHF): Guanarito virus- 브라질출혈열(Brazilian Hemorrhagic Fever, BzHF): Sabiá virus◦ 생존력: 건조한 환경에서 생존 불가능, 숙주 밖 혈액 검체에서 2주 정도 생존 가능◦ 소독 및 불활성화: 1% sodium hypochlorite, 2% glutaraldehyde, 10% formaldehyde, 55℃에서 30분 이상 가열, 121℃에서 15분 이상 고압증기멸균, 자외선 조사, 감마선 조사병원소(감염원)◦ 병원소: 생쥐, 들쥐, 다람쥐 등 설치류- 아르헨티나출혈열: 드라이랜드저녁쥐(Drylands vesper mouse, Calomys musculinus), 작은저녁쥐(Small vesper mouse, Calomys laucha), 아자라풀밭쥐(Azara’s akodant, Akodon azarae)- 볼리비아출혈열:  큰저녁쥐(Large vesper mouse, Calomys callosus)- 베네수엘라출혈열: 짧은꼬리사탕수수쥐(Short

In [44]:
# import requests

# for url in urls:
#     try:
#         response = requests.get(url)
#         if response.status_code == 200:
#             print(f"Success: {url}")
#         else:
#             print(f"Failed: {url} - Status Code: {response.status_code}")
#     except Exception as e:
#         print(f"Error: {url} - {e}")


## Chroma DB 생성

In [45]:
# persistant 한 DB의 path를 지정해줌
DB_path = './chroma_db'

# from_documents() 를 통해서 앞서 로드하고, 스플릿한 데이터를 db에 저장합니다.
# 문서를 디스크에 저장합니다. 저장시 persist_directory에 저장할 경로를 지정합니다.
# persist_directory 를 지정해줘야지 컴퓨터를 껐다가 다시 켜도 DB 가 저장이 되어있음.
persist_db = Chroma.from_documents(
    docs, OpenAIEmbeddings(model="text-embedding-3-large"), persist_directory=DB_path, collection_name = 'my_db'
)

In [5]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from dotenv import load_dotenv
from langchain.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_chroma import Chroma


# 환경 변수 로드
load_dotenv()

DB_path = './chroma_db'

embeddings = OpenAIEmbeddings(model = "text-embedding-3-large")  # OpenAI API 키는 .env 파일에서 로드됨

db = Chroma(
        persist_directory= DB_path,
        embedding_function=embeddings,
        collection_name='my_db'
        )
retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": 3})

llm = ChatOpenAI(
        model="gpt-4o",  # OpenAI 모델 선택
        temperature=0,  # 출력의 일관성을 높이기 위해 온도 설정
        max_tokens=1024)

prompt_template = """
You are an expert in infectious diseases. Provide accurate answers based strictly on the given context.

Instructions:
- Answer only using the provided context. Do not include creative ideas or answers not directly related to the question.
- If the question is outside the context, respond with: "I don't know."
- Include references in the "References" section using the source's URL from the metadata.
- Answer in Korean

#Example Format:
    (detailed answer to the question)

    **출처**
    - (URL of the source)

#Context:
{context}

#Question:
{question}

#Answer:
"""
prompt = PromptTemplate(input_variables=["context", "question"], template=prompt_template)

# 5. LLM 체인 생성
llm_chain = prompt|llm

# 6. RAG 체인 구성
rag_chain = {"context": retriever, "question": RunnablePassthrough()} | llm_chain |StrOutputParser()

In [6]:
question = '남아메리카 출혈열의 발생원인은'
response = rag_chain.invoke(question)
print(response)

남아메리카 출혈열의 발생원인은 남아메리카에서 발생한 출혈열 바이러스 감염에 의한 것입니다. 각 국가별로 다른 바이러스와 매개 설치류에 의해 발생하며, 아르헨티나출혈열은 Junín virus, 볼리비아출혈열은 Machupo virus, 베네수엘라출혈열은 Guanarito virus, 브라질출혈열은 Sabiá virus에 의해 발생합니다. 이 바이러스들은 주로 설치류에 의해 전파됩니다.

**출처**
- https://010601.busancidc-html.com


## FAISS DB 생성

In [49]:
from langchain_community.vectorstores import FAISS

db_faiss = FAISS.from_documents(documents = docs, embedding = OpenAIEmbeddings(model='text-embedding-3-large'))
db_faiss.save_local(folder_path = 'faiss_db', index_name='faiss_index')

In [3]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from dotenv import load_dotenv
from langchain.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


# 환경 변수 로드
load_dotenv()

folder_path = "./faiss_db"
pkl_path = "./faiss_db/faiss_index.pkl"


embeddings = OpenAIEmbeddings(model = "text-embedding-3-large")  # OpenAI API 키는 .env 파일에서 로드됨

db = FAISS.load_local(
        folder_path=folder_path,
        embeddings=embeddings,
        index_name = 'faiss_index',
        allow_dangerous_deserialization=True
    )
retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": 3})

llm = ChatOpenAI(
        model="gpt-4o",  # OpenAI 모델 선택
        temperature=0,  # 출력의 일관성을 높이기 위해 온도 설정
        max_tokens=1024)

prompt_template = """
You are an expert in infectious diseases. Provide accurate answers based strictly on the given context.

Instructions:
- Answer only using the provided context. Do not include creative ideas or answers not directly related to the question.
- If the question is outside the context, respond with: "I don't know."
- Include references in the "References" section using the source's URL from the metadata.
- Answer in Korean

#Example Format:
    (detailed answer to the question)

    **출처**
    - (URL of the source)

#Context:
{context}

#Question:
{question}

#Answer:
"""
prompt = PromptTemplate(input_variables=["context", "question"], template=prompt_template)

# 5. LLM 체인 생성
llm_chain = prompt|llm

# 6. RAG 체인 구성
rag_chain = {"context": retriever, "question": RunnablePassthrough()} | llm_chain |StrOutputParser()

In [4]:
question = '남아메리카 출혈열 발생원인은?'
response = rag_chain.invoke(question)
print(response)

남아메리카 출혈열의 발생 원인은 각 국가별로 다른 바이러스와 매개 설치류에 의해 발생합니다. 아르헨티나출혈열은 Junín 바이러스, 볼리비아출혈열은 Machupo 바이러스, 베네수엘라출혈열은 Guanarito 바이러스, 브라질출혈열은 Sabiá 바이러스에 의해 발생합니다. 이 바이러스들은 주로 설치류에 의해 전파됩니다.

**출처**
- https://010601.busancidc-html.com


### response 찍어보면 두 벡터 DB 모두 유사한 성능을 보임 (답변 출력에 걸리는 시간의 차이가 미비함.)
---

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from dotenv import load_dotenv
from langchain.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


# 환경 변수 로드
load_dotenv()

folder_path = "./data"
pkl_path = "./data/faiss_index.pkl"


embeddings = OpenAIEmbeddings(model = "text-embedding-3-large")  # OpenAI API 키는 .env 파일에서 로드됨

db = FAISS.load_local(
        folder_path=folder_path,
        embeddings=embeddings,
        index_name = 'faiss_index',
        allow_dangerous_deserialization=True
    )
retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": 3})

llm = ChatOpenAI(
        model="gpt-4o",  # OpenAI 모델 선택
        temperature=0,  # 출력의 일관성을 높이기 위해 온도 설정
        max_tokens=1024)

prompt_template = """
You are an expert in infectious diseases. Provide accurate answers based strictly on the given context.

Instructions:
- Answer only using the provided context. Do not include creative ideas or answers not directly related to the question.
- If the question is outside the context, respond with: "I don't know."
- Include references in the "References" section using the source's URL from the metadata.
- Answer in Korean

#Example Format:
    (detailed answer to the question)

    **출처**
    - (URL of the source)

#Context:
{context}

#Question:
{question}

#Answer:
"""
prompt = PromptTemplate(input_variables=["context", "question"], template=prompt_template)

# 5. LLM 체인 생성
llm_chain = prompt|llm

# 6. RAG 체인 구성
rag_chain = {"context": retriever, "question": RunnablePassthrough()} | llm_chain |StrOutputParser()